# Phương pháp lặp Jacobi

Giải hệ phương trình tuyến tính $Ax=b$, khởi tạo $x^{(0)}=0$.

$$x_i^{(k+1)}=\frac{b_i-\sum_{j\ne i}a_{ij}x_j^{(k)}}{a_{ii}}.$$

Khác với Gauss–Seidel, Jacobi chỉ dùng các giá trị ở lần lặp trước để tính toàn bộ nghiệm mới.

Đặt $q=\max_i\sum_{j\ne i}|a_{ij}|/|a_{ii}|$. Điều kiện $q<1$ là điều kiện đủ để hội tụ; nếu không đạt, chưa thể kết luận phương pháp không hội tụ. Notebook này chỉ thực hiện lặp khi điều kiện đủ được thỏa mãn.

Ước lượng sai số theo chuẩn vô cùng: $\|x^*-x^{(k+1)}\|_\infty \le \frac{q}{1-q}\|x^{(k+1)}-x^{(k)}\|_\infty$. Dừng khi ước lượng này nhỏ hơn `esl` hoặc đã đạt `max_blap`.

In [ ]:
import numpy as np


def Jacobi(A, b, max_blap, esl):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    if b.ndim != 1 or b.size == 0 or A.shape != (b.size, b.size):
        raise ValueError('A phải là ma trận vuông và b là vector cùng số hàng.')
    if not np.all(np.isfinite(A)) or not np.all(np.isfinite(b)):
        raise ValueError('A và b phải chứa các giá trị hữu hạn.')
    if np.any(np.diag(A) == 0):
        raise ValueError('Các phần tử trên đường chéo chính phải khác 0.')
    if not isinstance(max_blap, (int, np.integer)) or max_blap <= 0:
        raise ValueError('Số lần lặp tối đa phải là số nguyên dương.')
    if not np.isfinite(esl) or esl <= 0:
        raise ValueError('Sai số cho phép phải là số dương hữu hạn.')

    n = len(b)
    x = np.zeros(n)  # Khởi tạo vector x với giá trị 0

    # Kiểm tra điều kiện đủ để hội tụ và tìm q lớn nhất
    q_values = []
    for i in range(n):
        sigma = np.sum(np.abs(A[i, :i])) + np.sum(np.abs(A[i, i + 1:]))
        q = sigma / np.abs(A[i, i])
        q_values.append(q)
        print(f'q{i + 1} = {q}')

    t = max(q_values)
    print('Giá trị q lớn nhất:', t)
    htu = t < 1
    print('Điều kiện đủ để hội tụ:', htu)
    if not htu:
        print('Chưa bảo đảm hội tụ theo điều kiện q < 1. Dừng thuật toán.')
        return None

    print()
    for k in range(max_blap):
        x_old = np.copy(x)

        for i in range(n):
            # Cả hai tổng đều dùng nghiệm ở lần lặp trước
            sum1 = np.dot(A[i, :i], x_old[:i])
            sum2 = np.dot(A[i, i + 1:], x_old[i + 1:])
            x[i] = (b[i] - sum1 - sum2) / A[i, i]

        saiso = (t / (1 - t)) * np.max(np.abs(x - x_old))
        print(f'KQ lần lặp thứ {k + 1}: {x} \n   Ước lượng sai số: {saiso}')
        if saiso < esl:
            print(f'Hội tụ đạt được sau {k + 1} lần lặp.')
            return x

    print('Đã đạt số lần lặp tối đa; chưa đạt sai số cho phép.')
    return x


In [ ]:
# Ví dụ giống notebook Gauss–Seidel: thực hiện tối đa 3 lần lặp
A = np.array([[5, -1, 2],
              [1, 4, -1],
              [1, 1, 4]], dtype=float)
b = np.array([8, -4, 4], dtype=float)

kq_3 = Jacobi(A, b, max_blap=3, esl=1e-5)


In [ ]:
# Tăng số lần lặp tối đa để đạt sai số cho phép
kq = Jacobi(A, b, max_blap=100, esl=1e-5)
print('Nghiệm gần đúng:', kq)
